## 문서 파서 데모

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from document_processor import (
    ApplyTextEditsRequest,
    DocIR,
    DocumentInput,
    GetDocumentContextRequest,
    ListEditableTargetsRequest,
    RenderReviewHtmlRequest,
    TextAnnotation,
    TextEdit,
    apply_text_edits,
    get_document_context,
    list_editable_targets,
    render_review_html,
)
from doc_processor import ParagraphCategory, ParserConfig, RelevanceMode, run_parser

PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = PROJECT_DIR.parent

TESTS_DIR = PROJECT_DIR / "tests"

SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "01. 대중문화예술분야 연습생 표준계약서.hwpx"
# SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "02. 청소년 대중문화예술인 표준 부속합의서.hwpx"
# SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "style_test_sample.docx"

OUTPUT_DIR = TESTS_DIR / "results" / "docir_api_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SAMPLE_DOC.exists(), SAMPLE_DOC
SAMPLE_DOC_TYPE = SAMPLE_DOC.suffix.lower().lstrip(".")
SAMPLE_WRITEBACK_SUFFIX = ".hwpx" if SAMPLE_DOC_TYPE == "hwp" else SAMPLE_DOC.suffix.lower()


def anchor_summary(node):
    anchor = getattr(node, "native_anchor", None)
    if anchor is None:
        return None
    return anchor.model_dump(mode="json")


def paragraph_style_summary(paragraph):
    style = paragraph.para_style
    if style is None:
        return None
    style_dump = style.model_dump(mode="json")
    if style_dump.get("column_layout") is None and not any(
        value is not None for key, value in style_dump.items() if key != "column_layout"
    ):
        return None
    return style_dump


def pick_demo_paragraph(doc: DocIR):
    for paragraph in doc.paragraphs:
        if paragraph.runs and not paragraph.tables and not paragraph.images and paragraph.text.strip():
            for run in paragraph.runs:
                if run.text.strip():
                    return paragraph, run
    raise ValueError("Could not find a simple text paragraph/run pair for the demo.")


def iter_table_cells(table):
    for cell in table.cells:
        yield cell
        for paragraph in cell.paragraphs:
            for nested_table in paragraph.tables:
                yield from iter_table_cells(nested_table)


def iter_cells(doc: DocIR):
    for paragraph in doc.paragraphs:
        for table in paragraph.tables:
            yield from iter_table_cells(table)


def pick_demo_cell(doc: DocIR):
    for cell in iter_cells(doc):
        if not cell.text.strip() or not cell.paragraphs:
            continue
        if all(paragraph.runs and not paragraph.tables and not paragraph.images for paragraph in cell.paragraphs):
            return cell
    return None


def cell_by_id(doc: DocIR, node_id: str):
    for cell in iter_cells(doc):
        if cell.node_id == node_id:
            return cell
    raise KeyError(node_id)


def paragraph_by_id(doc: DocIR, node_id: str):
    for paragraph in doc.paragraphs:
        if paragraph.node_id == node_id:
            return paragraph
    raise KeyError(node_id)


def run_by_id(doc: DocIR, node_id: str):
    for paragraph in doc.paragraphs:
        for run in paragraph.runs:
            if run.node_id == node_id:
                return run
    raise KeyError(node_id)


PARSER_CATEGORY_COLORS = {
    ParagraphCategory.TITLE: "#D9EAD3",
    ParagraphCategory.PREAMBLE: "#FFF2CC",
    ParagraphCategory.CLAUSE_HEADING: "#F4CCCC",
    ParagraphCategory.CLAUSE_BODY: "#FCE5CD",
    ParagraphCategory.SUBCLAUSE_HEADING: "#D9D2E9",
    ParagraphCategory.SUBCLAUSE_BODY: "#CFE2F3",
    ParagraphCategory.INPUT_BLOCK: "#D0E0E3",
    ParagraphCategory.APPENDIX: "#EAD1DC",
    ParagraphCategory.HEADER: "#EFEFEF",
    ParagraphCategory.FOOTER: "#EFEFEF",
    ParagraphCategory.OTHER: "#E6E6E6",
    ParagraphCategory.BOUNDARY_SUSPECT: "#F9CB9C",
}


def parser_category_label(category: ParagraphCategory | None) -> str:
    if category is None:
        return "Unlabeled"
    return category.value.replace("_", " ").title()


def parser_category_color(category: ParagraphCategory | None) -> str:
    if category is None:
        return "#E6E6E6"
    return PARSER_CATEGORY_COLORS.get(category, "#E6E6E6")


def occurrence_index_for_span(text: str, selected_text: str, start: int) -> int | None:
    matches = []
    search_from = 0
    while True:
        index = text.find(selected_text, search_from)
        if index < 0:
            break
        matches.append(index)
        search_from = index + 1
    if not matches:
        raise ValueError(f"Could not find {selected_text!r} in paragraph text.")
    occurrence_index = matches.index(start)
    return occurrence_index if len(matches) > 1 else None


def parser_annotations_from_doc(doc: DocIR):
    annotations = []
    skipped = []
    for paragraph in doc.paragraphs:
        parser_meta = paragraph.meta.parser if paragraph.meta and paragraph.meta.parser else None
        if parser_meta is None or not paragraph.text.strip():
            continue
        if paragraph.tables or paragraph.images:
            skipped.append(paragraph.node_id)
            continue

        if parser_meta.spans:
            for span in parser_meta.spans:
                selected_text = paragraph.text[span.start:span.end]
                if not selected_text:
                    continue
                occurrence_index = occurrence_index_for_span(paragraph.text, selected_text, span.start)
                note_parts = [f"category={span.kind.value}"]
                if span.clause_no:
                    note_parts.append(f"clause={span.clause_no}")
                if span.subclause_no:
                    note_parts.append(f"subclause={span.subclause_no}")
                annotations.append(
                    TextAnnotation(
                        target_kind="paragraph",
                        target_id=paragraph.node_id,
                        selected_text=selected_text,
                        occurrence_index=occurrence_index,
                        label=parser_category_label(span.kind),
                        color=parser_category_color(span.kind),
                        note=" | ".join(note_parts),
                    )
                )
            continue

        if parser_meta.category is None:
            continue

        note_parts = [f"category={parser_meta.category.value}"]
        if parser_meta.clause_no:
            note_parts.append(f"clause={parser_meta.clause_no}")
        if parser_meta.subclause_no:
            note_parts.append(f"subclause={parser_meta.subclause_no}")
        if parser_meta.boundary_suspect:
            note_parts.append("boundary_suspect=true")
        annotations.append(
            TextAnnotation(
                target_kind="paragraph",
                target_id=paragraph.node_id,
                label=parser_category_label(parser_meta.category),
                color=parser_category_color(parser_meta.category),
                note=" | ".join(note_parts),
            )
        )

    return annotations, skipped

SAMPLE_DOC


PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/new_test/01. 대중문화예술분야 연습생 표준계약서.hwpx')

### 파일 경로나 바이너리 통해서 적제하기

`DocIR.from_file(...)` 통해서 적제 가능

In [2]:
sample_bytes = SAMPLE_DOC.read_bytes()

doc_from_path = DocIR.from_file(SAMPLE_DOC)
doc_from_bytes = DocIR.from_file(sample_bytes, doc_type=SAMPLE_DOC_TYPE)

demo_paragraph, demo_run = pick_demo_paragraph(doc_from_path)
demo_cell = pick_demo_cell(doc_from_path)

summary = {
    "sample_doc": str(SAMPLE_DOC.relative_to(TESTS_DIR)),
    "source_doc_type": doc_from_path.source_doc_type,
    "paragraph_count": len(doc_from_path.paragraphs),
    "page_count": len(doc_from_path.pages),
    "path_vs_bytes_same_first_paragraph": doc_from_path.paragraphs[0].text == doc_from_bytes.paragraphs[0].text,
    "demo_paragraph_node_id": demo_paragraph.node_id,
    "demo_paragraph_native_anchor": anchor_summary(demo_paragraph),
    "demo_paragraph_style": paragraph_style_summary(demo_paragraph),
    "demo_run_node_id": demo_run.node_id,
    "demo_run_native_anchor": anchor_summary(demo_run),
    "demo_cell_node_id": demo_cell.node_id if demo_cell else None,
    "demo_cell_text": demo_cell.text if demo_cell else None,
}

first_paragraphs = [
    {
        "node_id": paragraph.node_id,
        "text": paragraph.text,
        "run_ids": [run.node_id for run in paragraph.runs],
        "native_anchor": anchor_summary(paragraph),
        "para_style": paragraph_style_summary(paragraph),
    }
    for paragraph in doc_from_path.paragraphs[:3]
]

summary, first_paragraphs


({'sample_doc': 'doc_samples/new_test/01. 대중문화예술분야 연습생 표준계약서.hwpx',
  'source_doc_type': 'hwpx',
  'paragraph_count': 120,
  'page_count': 6,
  'path_vs_bytes_same_first_paragraph': True,
  'demo_paragraph_node_id': 'p_bfa4da5a775f2ffd',
  'demo_paragraph_native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'paragraph',
   'debug_path': 's1.p1',
   'parent_debug_path': None,
   'part_name': 'Contents/section0.xml',
   'structural_path': 's1.p1',
   'text_hash': '8eba3ed12a3b89fed81a0580ea6380b29e3a38c5'},
  'demo_paragraph_style': {'align': 'justify',
   'left_indent_pt': 0.0,
   'right_indent_pt': 0.0,
   'first_line_indent_pt': 0.0,
   'hanging_indent_pt': None,
   'column_layout': {'count': 1,
    'gap_pt': 0.0,
    'widths_pt': [],
    'gaps_pt': [],
    'equal_width': True}},
  'demo_run_node_id': 'r_3f1ff7241702452b',
  'demo_run_native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'run',
   'debug_path': 's1.p1.r2',
   'parent_debug_path': 's1.p1',
   'part_name': 

### (LLM 대상) 주변 context/문맥 보기 및 안전한 수정 구간 선정

`DocIR`가 사용하는 stateless API는 문서 레퍼런스로 `DocumentInput`를 공통으로 사용함

문서 경로나 `DocIR`를 직접 넣을 수 있음!

현재 API에서 LLM이 잡고 수정해야 하는 안정적인 식별자는 `node_id`이고, 원본 위치 확인용 정보는 `native_anchor`로 확인함.


In [3]:
# 타겟 기준 뒷부분 한 문단 가져오기 -> sliding window같은 문맥 확보 가능
context = get_document_context(
    GetDocumentContextRequest(
        document=DocumentInput(source_path=str(SAMPLE_DOC)),  # 문서 경로 바로 적재
        target_ids=[demo_run.node_id],
        before=0,
        after=1,
        include_runs=True,
    )
)

# LLM이나 기타 자동화 도구들이 가져와 수정할 수 있는 "안전한" 후보군 가져오기
targets = list_editable_targets(
    ListEditableTargetsRequest(
        document=DocumentInput(doc_ir=doc_from_path),  # DocIR 직접 주입
        target_ids=[demo_paragraph.node_id],
        target_kinds=["paragraph", "run"],
        include_child_runs=True,
    )
)

context_summary = [
    {
        "node_id": paragraph.node_id,
        "text": paragraph.text,
        "native_anchor": anchor_summary(paragraph),
        "runs": [(run.node_id, run.text) for run in paragraph.runs],
    }
    for paragraph in context.paragraphs
]

target_summary = [
    {
        "target_kind": target.target_kind,
        "target_id": target.target_id,
        "current_text": target.current_text,
        "native_anchor": target.native_anchor.model_dump(mode="json") if target.native_anchor else None,
        "writable": target.writable,
    }
    for target in targets.targets
]

print("ctx summary\n", "="*25)
display(context_summary)
print("target summary\n", "="*25)
display(target_summary)


ctx summary


[{'node_id': 'p_bfa4da5a775f2ffd',
  'text': '[별표1]',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'paragraph',
   'debug_path': 's1.p1',
   'parent_debug_path': None,
   'part_name': 'Contents/section0.xml',
   'structural_path': 's1.p1',
   'text_hash': '8eba3ed12a3b89fed81a0580ea6380b29e3a38c5'},
  'runs': [('r_f017369d1a2e2a89', ''), ('r_3f1ff7241702452b', '[별표1]')]},
 {'node_id': 'p_b137ee595f9f9767',
  'text': '대중문화예술분야 연습생 표준계약서',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'paragraph',
   'debug_path': 's1.p2',
   'parent_debug_path': None,
   'part_name': 'Contents/section0.xml',
   'structural_path': 's1.p2',
   'text_hash': '7a2e7bbcd62f3968ab8dae0f475505e15c381244'},
  'runs': [('r_70a2dd7d5d150def', '대중문화예술분야 연습생 표준계약서')]}]

target summary


[{'target_kind': 'paragraph',
  'target_id': 'p_bfa4da5a775f2ffd',
  'current_text': '[별표1]',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'paragraph',
   'debug_path': 's1.p1',
   'parent_debug_path': None,
   'part_name': 'Contents/section0.xml',
   'structural_path': 's1.p1',
   'text_hash': '8eba3ed12a3b89fed81a0580ea6380b29e3a38c5'},
  'writable': True},
 {'target_kind': 'run',
  'target_id': 'r_f017369d1a2e2a89',
  'current_text': '',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'run',
   'debug_path': 's1.p1.r1',
   'parent_debug_path': 's1.p1',
   'part_name': None,
   'structural_path': 's1.p1.r1',
   'text_hash': 'da39a3ee5e6b4b0d3255bfef95601890afd80709'},
  'writable': True},
 {'target_kind': 'run',
  'target_id': 'r_3f1ff7241702452b',
  'current_text': '[별표1]',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'run',
   'debug_path': 's1.p1.r2',
   'parent_debug_path': 's1.p1',
   'part_name': None,
   'structural_path': 's1

### 표/셀 편집 대상 보기

`target_kind="cell"`을 사용하면 표 안의 셀 전체 텍스트를 편집 대상으로 볼 수 있음. `include_child_runs=True`를 같이 주면 같은 셀 안의 run들도 같이 확인 가능.


In [4]:
if demo_cell is None:
    cell_target_summary = [
        {
            "message": "No simple editable table cell was found in the selected sample document.",
        }
    ]
else:
    cell_targets = list_editable_targets(
        ListEditableTargetsRequest(
            document=DocumentInput(doc_ir=doc_from_path),
            target_ids=[demo_cell.node_id],
            target_kinds=["cell", "run"],
            include_child_runs=True,
            only_writable=False,
        )
    )
    cell_target_summary = [
        {
            "target_kind": target.target_kind,
            "target_id": target.target_id,
            "current_text": target.current_text,
            "native_anchor": target.native_anchor.model_dump(mode="json") if target.native_anchor else None,
            "writable": target.writable,
            "writable_reason": target.writable_reason,
        }
        for target in cell_targets.targets
    ]

print("cell target summary\n", "="*25)
display(cell_target_summary)


cell target summary


[{'target_kind': 'cell',
  'target_id': 'cell_c140d37cf7f53629',
  'current_text': '문화체육관광부고시 \n제2025-0069호',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'cell',
   'debug_path': 's1.p3.r1.tbl1.tr2.tc1',
   'parent_debug_path': 's1.p3.r1.tbl1',
   'part_name': None,
   'structural_path': 's1.p3.r1.tbl1.tr2.tc1',
   'text_hash': '76954d09be81455cc49a054c8e4faf03c3a26ac6'},
  'writable': True,
  'writable_reason': None},
 {'target_kind': 'run',
  'target_id': 'r_f3a5f24dd9940885',
  'current_text': '문화체육관광부고시 ',
  'native_anchor': {'source_doc_type': 'hwpx',
   'node_kind': 'run',
   'debug_path': 's1.p3.r1.tbl1.tr2.tc1.p1.r1',
   'parent_debug_path': 's1.p3.r1.tbl1.tr2.tc1.p1',
   'part_name': None,
   'structural_path': 's1.p3.r1.tbl1.tr2.tc1.p1.r1',
   'text_hash': '3d8d3972bd6d5c74ad86dd41f40f2ad81e99c5f0'},
  'writable': True,
  'writable_reason': None},
 {'target_kind': 'run',
  'target_id': 'r_4498f11395882619',
  'current_text': '제2025-0069호',
  'native_anchor'

### HTML preview 렌더링하기

`DocIR.to_html(...)` 사용하면 자체 엔진이 HTML로 페이지를 재현함, `render_review_html(...)` 추가 기능은 `TextAnnotation`을 같이 넣어주면 하이라이팅을 해줄 수 있음.

`occurrence_index`를 사용해서 필요한 run 안의 수정구간을 명시 -> 여기서 "**중요한 부분**"을 하이라이팅


In [5]:
paragraph_selection = demo_paragraph.text[: min(20, len(demo_paragraph.text))]
run_selection = demo_run.text[: min(8, len(demo_run.text))]

# 원본 HTML 출력
preview_html = doc_from_path.to_html(title="DocIR Preview")
preview_path = OUTPUT_DIR / "preview.html"
preview_path.write_text(preview_html, encoding="utf-8")

# Annotate할 경우 텍스트 출력 - HTML
review = render_review_html(
    RenderReviewHtmlRequest(
        document=DocumentInput(doc_ir=doc_from_path),
        annotations=[
            TextAnnotation(
                target_kind="paragraph",
                target_id=demo_paragraph.node_id,
                selected_text=paragraph_selection,
                label="Paragraph span",
                color="#FFD966",
                note="Paragraph-level review anchor",
            ),
            TextAnnotation(
                target_kind="run",
                target_id=demo_run.node_id,
                selected_text=run_selection,
                label="Run span",
                color="#99EEFF",
                note="Run-level review anchor",
            ),
        ],
        title="DocIR Review Demo",
    )
)

# 예시 문서(from_mapping): 등장 횟수 기준 TextAnnotation 방식
ambiguous_doc = DocIR.from_mapping({"s1.p1.r1": "Beta Beta Beta"}, source_doc_type="docx")
ambiguous_run = ambiguous_doc.paragraphs[0].runs[0]
ambiguous_review = render_review_html(
    RenderReviewHtmlRequest(
        document=DocumentInput(doc_ir=ambiguous_doc),
        annotations=[
            TextAnnotation(
                target_kind="run",
                target_id=ambiguous_run.node_id,
                selected_text="Beta",
                occurrence_index=1,  # Zero index 기준
                label="Second Beta",
                color="#C9DAF8",
                note="Repeated text uses occurrence_index, not raw offsets.",
            )
        ],
        title="Repeated Text Demo",
    )
)

review_path = OUTPUT_DIR / "review.html"
review_path.write_text(review.html or "", encoding="utf-8")

print("annotation HTML res.:\n", "="*25)
display([item.model_dump(mode="json") for item in review.resolved_annotations])
print("ambiguous (중복) 있는경우:\n", "="*25)
display([item.model_dump(mode="json") for item in ambiguous_review.resolved_annotations])


annotation HTML res.:


[{'target_kind': 'paragraph',
  'target_id': 'p_bfa4da5a775f2ffd',
  'selected_text': '[별표1]',
  'occurrence_index': 0,
  'start': 0,
  'end': 5,
  'label': 'Paragraph span',
  'color': '#FFD966',
  'note': 'Paragraph-level review anchor'},
 {'target_kind': 'run',
  'target_id': 'r_3f1ff7241702452b',
  'selected_text': '[별표1]',
  'occurrence_index': 0,
  'start': 0,
  'end': 5,
  'label': 'Run span',
  'color': '#99EEFF',
  'note': 'Run-level review anchor'}]

ambiguous (중복) 있는경우:


[{'target_kind': 'run',
  'target_id': 'r_f017369d1a2e2a89',
  'selected_text': 'Beta',
  'occurrence_index': 1,
  'start': 5,
  'end': 9,
  'label': 'Second Beta',
  'color': '#C9DAF8',
  'note': 'Repeated text uses occurrence_index, not raw offsets.'}]

근데... 왜 occurence index(N번째 등장 단어)를 사용?
그냥 string index(N번째 글자)를 쓰면 되지 않나...?

언어모델은 상세한 숫자세기를 잘 못하고, 내어쓰기나 이상한 빈칸 있을 경우 오해할 여지가 있음! "N번째 단어" 형식으로 답변을 하게 하는 것이 최적!

### 수정사항 반영하기

- `DocIR`만 써서 in-memory 수정/저장
- 바로 파일에 적용
- 바이너리(바이트)로 바로 적용 (컨텍스트 매니저나 파일 포인터 등)

모든 수정 요청은 `target_kind`, `target_id`, `expected_text`, `new_text`를 함께 넘김. `expected_text`가 현재 문서와 다르면 수정이 거절되므로 LLM tool calling에서도 비교적 안전하게 사용할 수 있음.

> !! hwp를 수정해서 저장하면 hwpx로 저장 !!


In [6]:
# DocIR안에 직접 저장 (in-memory)
in_memory_result = apply_text_edits(
    ApplyTextEditsRequest(
        document=DocumentInput(doc_ir=doc_from_path),
        edits=[
            TextEdit(
                target_kind="paragraph",
                target_id=demo_paragraph.node_id,
                expected_text=demo_paragraph.text,
                new_text=f"{demo_paragraph.text} [DocIR demo edit]",
                reason="Show in-memory paragraph editing",
            )
        ],
    )
)

# 파일에 수정해서 바로 저장 (on disk)
native_output = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_native_edit{SAMPLE_WRITEBACK_SUFFIX}"
native_result = apply_text_edits(
    ApplyTextEditsRequest(
        document=DocumentInput(source_path=str(SAMPLE_DOC)),
        edits=[
            TextEdit(
                target_kind="run",
                target_id=demo_run.node_id,
                expected_text=demo_run.text,
                new_text=f"[native edited here -> {demo_run.text.rstrip()} <- test]",
                reason="Show native file write-back",
            )
        ],
        output_path=str(native_output),
        return_doc_ir=True,
    )
)

# 바이너리로 바로 저장
bytes_result = apply_text_edits(
    ApplyTextEditsRequest(
        document=DocumentInput(source_bytes=sample_bytes, source_name=SAMPLE_DOC.name),
        edits=[
            TextEdit(
                target_kind="run",
                target_id=demo_run.node_id,
                expected_text=demo_run.text,
                new_text=f"{demo_run.text.rstrip()} [bytes]",
                reason="Show bytes-backed write-back",
            )
        ],
        return_doc_ir=True,
    )
)

# 셀 전체 텍스트 편집: paragraph/run과 같은 API에서 target_kind="cell" 사용
if demo_cell is None:
    cell_edit_summary = {
        "message": "No simple editable table cell was found in the selected sample document.",
    }
else:
    cell_demo_new_text = f"{demo_cell.text.rstrip()} [cell demo edit]"

    cell_in_memory_result = apply_text_edits(
        ApplyTextEditsRequest(
            document=DocumentInput(doc_ir=doc_from_path),
            edits=[
                TextEdit(
                    target_kind="cell",
                    target_id=demo_cell.node_id,
                    expected_text=demo_cell.text,
                    new_text=cell_demo_new_text,
                    reason="Show in-memory table cell editing",
                )
            ],
        )
    )

    cell_native_output = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_cell_native_edit{SAMPLE_WRITEBACK_SUFFIX}"
    cell_native_result = apply_text_edits(
        ApplyTextEditsRequest(
            document=DocumentInput(source_path=str(SAMPLE_DOC)),
            edits=[
                TextEdit(
                    target_kind="cell",
                    target_id=demo_cell.node_id,
                    expected_text=demo_cell.text,
                    new_text=cell_demo_new_text,
                    reason="Show native table cell write-back",
                )
            ],
            output_path=str(cell_native_output),
            return_doc_ir=True,
        )
    )

    cell_bytes_result = apply_text_edits(
        ApplyTextEditsRequest(
            document=DocumentInput(source_bytes=sample_bytes, source_name=SAMPLE_DOC.name),
            edits=[
                TextEdit(
                    target_kind="cell",
                    target_id=demo_cell.node_id,
                    expected_text=demo_cell.text,
                    new_text=cell_demo_new_text,
                    reason="Show bytes-backed table cell write-back",
                )
            ],
            return_doc_ir=True,
        )
    )

    cell_edit_summary = {
        "cell_target_id": demo_cell.node_id,
        "before_text": demo_cell.text,
        "in_memory_updated_cell": cell_by_id(cell_in_memory_result.updated_doc_ir, demo_cell.node_id).text,
        "native_cell_output_path": str(Path(cell_native_result.output_path).relative_to(TESTS_DIR)),
        "native_updated_cell": cell_by_id(cell_native_result.updated_doc_ir, demo_cell.node_id).text,
        "bytes_cell_output_filename": cell_bytes_result.output_filename,
        "bytes_updated_cell": cell_by_id(cell_bytes_result.updated_doc_ir, demo_cell.node_id).text,
    }

{
    "in_memory_updated_paragraph": paragraph_by_id(in_memory_result.updated_doc_ir, demo_paragraph.node_id).text,
    "native_output_path": str(Path(native_result.output_path).relative_to(TESTS_DIR)),
    "native_updated_run": run_by_id(native_result.updated_doc_ir, demo_run.node_id).text,
    "bytes_output_filename": bytes_result.output_filename,
    "bytes_output_size": len(bytes_result.output_bytes or b""),
    "bytes_updated_run": run_by_id(bytes_result.updated_doc_ir, demo_run.node_id).text,
    "cell_edit_summary": cell_edit_summary,
}


{'in_memory_updated_paragraph': '[별표1] [DocIR demo edit]',
 'native_output_path': 'results/docir_api_demo/01. 대중문화예술분야 연습생 표준계약서_native_edit.hwpx',
 'native_updated_run': '[native edited here -> [별표1] <- test]',
 'bytes_output_filename': '01. 대중문화예술분야 연습생 표준계약서_edited.hwpx',
 'bytes_output_size': 63757,
 'bytes_updated_run': '[별표1] [bytes]',
 'cell_edit_summary': {'cell_target_id': 'cell_c140d37cf7f53629',
  'before_text': '문화체육관광부고시 \n제2025-0069호',
  'in_memory_updated_cell': '문화체육관광부고시 \n제2025-0069호 [cell demo edit]',
  'native_cell_output_path': 'results/docir_api_demo/01. 대중문화예술분야 연습생 표준계약서_cell_native_edit.hwpx',
  'native_updated_cell': '문화체육관광부고시 \n제2025-0069호 [cell demo edit]',
  'bytes_cell_output_filename': '01. 대중문화예술분야 연습생 표준계약서_edited.hwpx',
  'bytes_updated_cell': '문화체육관광부고시 \n제2025-0069호 [cell demo edit]'}}

### 지능형 파서 통해서 문서 부문 라벨링

지능형 파서로 일반 파서(정규식 등)가 잡지 못한 상세 부분 정리 및 하이라이팅 후 HTML로 출력

In [7]:
parser_state = run_parser(
    SAMPLE_DOC,
    config=ParserConfig(
        relevance_mode=RelevanceMode.KEYWORD_THEN_LLM,
        boundary_review_enabled=True,
        label_review_enabled=True,
        langfuse_enabled=True,
    ),
)

parser_doc = parser_state.working_doc
parser_result = parser_state.parser_result
assert parser_doc is not None
assert parser_result is not None

parser_annotations, parser_skipped = parser_annotations_from_doc(parser_doc)
parser_category_counts = Counter(annotation.label for annotation in parser_annotations)

parser_review = render_review_html(
    RenderReviewHtmlRequest(
        document=DocumentInput(doc_ir=parser_doc),
        annotations=parser_annotations,
        title="Parser Category Review",
    )
)

parser_review_path = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_parser_categories.html"
parser_review_path.write_text(parser_review.html or "", encoding="utf-8")

parser_legend = {
    parser_category_label(category): color
    for category, color in PARSER_CATEGORY_COLORS.items()
}

{
    "sample_doc": str(SAMPLE_DOC.relative_to(TESTS_DIR)),
    "accepted": parser_result.accepted,
    "reason": parser_result.reason,
    "clause_count": parser_result.clause_count,
    "subclause_count": parser_result.subclause_count,
    "annotation_count": len(parser_annotations),
    "category_counts": dict(sorted(parser_category_counts.items())),
    "category_colors": parser_legend,
    "skipped_mixed_content_paragraph_count": len(parser_skipped),
    "skipped_mixed_content_examples": parser_skipped[:10],
    "parser_notes": list(parser_result.notes),
    "parser_review_path": str(parser_review_path.relative_to(TESTS_DIR)),
    "resolved_annotation_preview": [item.model_dump(mode="json") for item in parser_review.resolved_annotations[:12]],
}


2026-04-22 17:23:28,178 | INFO | structure analysis run start source=/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/new_test/01. 대중문화예술분야 연습생 표준계약서.hwpx
2026-04-22 17:23:28,261 | INFO | [structure_analysis.load_document] start
2026-04-22 17:23:28,271 | INFO | [structure_analysis.load_document] done goto=screen_relevance
2026-04-22 17:23:28,282 | INFO | [structure_analysis.screen_relevance] start
2026-04-22 17:23:28,283 | INFO | [structure_analysis.screen_relevance] done goto=regex_analysis
2026-04-22 17:23:28,285 | INFO | [structure_analysis.regex_analysis] start
2026-04-22 17:23:28,287 | INFO | [structure_analysis.regex_analysis] done goto=llm_analysis
2026-04-22 17:23:28,292 | INFO | [structure_analysis.llm_analysis] start
2026-04-22 17:23:28,293 | INFO | [structure_analysis.llm_analysis] dispatching boundary batch review (25 suspects)
2026-04-22 17:23:28,293 | INFO | [structure_analysis.llm_analysis] done goto=boundary_llm_batch
2026-04-22 17:23:28,296 | IN

{'sample_doc': 'doc_samples/new_test/01. 대중문화예술분야 연습생 표준계약서.hwpx',
 'accepted': True,
 'reason': 'Parser clause parsing completed.',
 'clause_count': 13,
 'subclause_count': 40,
 'annotation_count': 81,
 'category_counts': {'Appendix': 2,
  'Clause Heading': 15,
  'Input Block': 16,
  'Other': 1,
  'Preamble': 3,
  'Subclause Body': 2,
  'Subclause Heading': 40,
  'Title': 2},
 'category_colors': {'Title': '#D9EAD3',
  'Preamble': '#FFF2CC',
  'Clause Heading': '#F4CCCC',
  'Clause Body': '#FCE5CD',
  'Subclause Heading': '#D9D2E9',
  'Subclause Body': '#CFE2F3',
  'Input Block': '#D0E0E3',
  'Appendix': '#EAD1DC',
  'Header': '#EFEFEF',
  'Footer': '#EFEFEF',
  'Other': '#E6E6E6',
  'Boundary Suspect': '#F9CB9C'},
 'skipped_mixed_content_paragraph_count': 1,
 'skipped_mixed_content_examples': ['p_70a5402160bbf4b7'],
 'parser_notes': ["Detected clause rule 'article' and subclause rule 'circled'."],
 'parser_review_path': 'results/docir_api_demo/01. 대중문화예술분야 연습생 표준계약서_parser_categories.

Failed to export span batch due to timeout, max retries or shutdown.
